Notebook này sẽ tách **16 nhãn MBTI** thành **4 bài toán Nhị phân** độc lập:
1. Hướng ngoại (E) vs Hướng nội (I)
2. Thực tế (S) vs Trực giác (N)
3. Lý trí (T) vs Cảm xúc (F)
4. Nguyên tắc (J) vs Linh hoạt (P)

Đồng thời, chúng ta sử dụng **SMOTE** để cân bằng lại số lượng nhãn (để AI không đoán lệch sang nhóm người đông).


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier

# Thư viện chống mất cân bằng Class
from imblearn.over_sampling import SMOTE

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

print("Load complete")


Load complete


## 1. Tách Data thành 4 phân khúc MBTI
Cách này giúp tách 1 chữ MBTI (VD: `INTP`) thành 4 cột mục tiêu (`I`, `N`, `T`, `P`).


In [5]:
df = pd.read_csv('../data/mbti_master_training_data.csv')
df = df.dropna(subset=['mbti_label'])
df = df[df['mbti_label'].str.len() == 4]

# Tách chuỗi MBTI
df['target_EI'] = df['mbti_label'].apply(lambda x: 1 if x[0] == 'E' else 0)
df['target_SN'] = df['mbti_label'].apply(lambda x: 1 if x[1] == 'S' else 0)
df['target_TF'] = df['mbti_label'].apply(lambda x: 1 if x[2] == 'T' else 0)
df['target_JP'] = df['mbti_label'].apply(lambda x: 1 if x[3] == 'J' else 0)

# === FEATURE ENGINEERING ===
# Features gốc
base_features = [
    'spotify_popularity', 'release_year',
    'genre_ei_score', 'genre_sn_score', 'genre_tf_score',
    'tempo_bpm', 'energy', 'danceability',
    'spectral_centroid', 'spectral_flatness', 'lyrics_polarity'
]

# Xoá NaN ở features gốc
df = df.dropna(subset=base_features)

# Tạo features mới từ tổ hợp (Interaction Features)
df['energy_x_dance'] = df['energy'] * df['danceability']           # Năng lượng nhảy
df['energy_x_tempo'] = df['energy'] * df['tempo_bpm']              # Năng lượng x Tốc độ
df['spectral_ratio'] = df['spectral_centroid'] / (df['spectral_flatness'] + 1e-8)  # Tỷ lệ phổ
df['dance_x_polarity'] = df['danceability'] * df['lyrics_polarity']  # Nhảy x Cảm xúc lời
df['energy_minus_dance'] = df['energy'] - df['danceability']        # Chênh lệch
df['tempo_normalized'] = df['tempo_bpm'] / 200.0                    # Chuẩn hoá tempo
df['popularity_log'] = np.log1p(df['spotify_popularity'])           # Log popularity
df['genre_avg'] = (df['genre_ei_score'] + df['genre_sn_score'] + df['genre_tf_score']) / 3  # Trung bình genre
df['spectral_x_energy'] = df['spectral_centroid'] * df['energy']    # Phổ x Năng lượng
df['polarity_abs'] = df['lyrics_polarity'].abs()                    # Cường độ cảm xúc

# Danh sách TOÀN BỘ features
features = base_features + [
    'energy_x_dance', 'energy_x_tempo', 'spectral_ratio',
    'dance_x_polarity', 'energy_minus_dance', 'tempo_normalized',
    'popularity_log', 'genre_avg', 'spectral_x_energy', 'polarity_abs'
]

X = df[features]
print(f"Dataset: {X.shape[0]} bài hát, {X.shape[1]} features")
print(f"Features: {features}")
df[['mbti_label', 'target_EI', 'target_SN', 'target_TF', 'target_JP']].head(3)

Dataset: 5330 bài hát, 21 features
Features: ['spotify_popularity', 'release_year', 'genre_ei_score', 'genre_sn_score', 'genre_tf_score', 'tempo_bpm', 'energy', 'danceability', 'spectral_centroid', 'spectral_flatness', 'lyrics_polarity', 'energy_x_dance', 'energy_x_tempo', 'spectral_ratio', 'dance_x_polarity', 'energy_minus_dance', 'tempo_normalized', 'popularity_log', 'genre_avg', 'spectral_x_energy', 'polarity_abs']


,mbti_label,target_EI,target_SN,target_TF,target_JP
0,INTP,0,0,1,0
1,INTP,0,0,1,0
2,INTP,0,0,1,0


## 2. Huấn luyện đa mô hình (Multi-Model Training)
Chúng ta sẽ viết 1 hàm để Train tự động 4 Model XGBoost cho 4 trục!


## 2.2. Kiểm chứng chéo (Cross-Validation) & Overfitting Check
Ở bước này, chúng ta so sánh:
1. **Train Accuracy**: AI học tốt thế nào trên dữ liệu đã biết.
2. **Test Accuracy**: AI dự đoán tốt thế nào trên dữ liệu mới.
3. **Cross-Validation (CV)**: Chia dữ liệu làm 5 phần, luân phiên dùng 4 phần để học và 1 phần để thi. Đây là con số khách quan nhất để đánh giá mô hình.
Nếu **Train Accuracy** cao vọt hẳn so với **Test Accuracy** (>10%), đó chính là **Overfitting**.


In [ ]:
from sklearn.model_selection import cross_val_score, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from scipy.stats import randint, uniform
import warnings
import time
warnings.filterwarnings('ignore')

def train_mbti_dimension(X, y, dimension_name, positive_label, negative_label):
    X_np = X.values if hasattr(X, 'values') else X
    y_np = y.values if hasattr(y, 'values') else y
    
    X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42, stratify=y_np)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0)
    X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0)
    
    # SMOTE
    smote = SMOTE(random_state=42)
    X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
    
    start = time.time()
    
    # === HYPERPARAMETER TUNING (200 iterations, 10-fold) ===
    param_dist = {
        'n_estimators': randint(300, 1500),
        'max_depth': randint(3, 12),
        'learning_rate': uniform(0.005, 0.2),
        'subsample': uniform(0.5, 0.5),
        'colsample_bytree': uniform(0.5, 0.5),
        'min_child_weight': randint(1, 15),
        'gamma': uniform(0, 1.0),
        'reg_alpha': uniform(0, 2),
        'reg_lambda': uniform(0.5, 3),
        'scale_pos_weight': uniform(0.8, 1.5),
    }
    
    xgb_base = XGBClassifier(
        eval_metric='logloss',
        random_state=42,
        tree_method='hist',
    )
    
    cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    
    search = RandomizedSearchCV(
        xgb_base, param_dist,
        n_iter=200,
        cv=cv_strategy,
        scoring='accuracy',
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    search.fit(X_train_smote, y_train_smote)
    best_xgb = search.best_estimator_
    
    # RandomForest
    rf = RandomForestClassifier(
        n_estimators=500, max_depth=10, min_samples_split=5,
        min_samples_leaf=2, random_state=42, n_jobs=-1
    )
    rf.fit(X_train_smote, y_train_smote)
    
    # GradientBoosting
    gb = GradientBoostingClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, random_state=42
    )
    gb.fit(X_train_smote, y_train_smote)
    
    # === ENSEMBLE: XGBoost + RandomForest + GradientBoosting ===
    model = VotingClassifier(
        estimators=[('xgb', best_xgb), ('rf', rf), ('gb', gb)],
        voting='soft'
    )
    model.fit(X_train_smote, y_train_smote)
    
    elapsed = time.time() - start
    
    # --- ĐÁNH GIÁ ---
    train_preds = model.predict(X_train_smote)
    train_acc = accuracy_score(y_train_smote, train_preds)
    
    test_preds = model.predict(X_test_scaled)
    test_acc = accuracy_score(y_test, test_preds)
    
    X_all_scaled = np.nan_to_num(scaler.transform(X_np), nan=0.0)
    cv_scores = cross_val_score(best_xgb, X_all_scaled, y_np, cv=cv_strategy)
    
    print(f"\n{'='*55}")
    print(f" TRỤC [{dimension_name}] (⏱️ {elapsed:.1f}s)")
    print(f"{'='*55}")
    print(f"   Best params: {search.best_params_}")
    print(f"   - Train Accuracy: {train_acc*100:.2f}%")
    print(f"   - Test Accuracy:  {test_acc*100:.2f}%")
    print(f"   - CV Accuracy:    {cv_scores.mean()*100:.2f}% (+/- {cv_scores.std()*100:.2f}%)")
    
    gap = (train_acc - test_acc) * 100
    if gap > 10:
        print(f"CẢNH BÁO: Overfitting (Gap: {gap:.2f}%)")
    else:
        print(f"ỔN ĐỊNH (Gap: {gap:.2f}%)")
    
    return model, scaler, y_test, test_preds

print("TRAINING VỚI TUNING NẶNG (200 iter, 10-fold, 3-model ensemble)")
print("Mỗi trục mất ~2-5 phút. Tổng ~10-20 phút.\n")

total_start = time.time()
model_EI, scaler_EI, y_test_EI, preds_EI = train_mbti_dimension(X, df['target_EI'], "E/I", "E (1)", "I (0)")
model_SN, scaler_SN, y_test_SN, preds_SN = train_mbti_dimension(X, df['target_SN'], "S/N", "S (1)", "N (0)")
model_TF, scaler_TF, y_test_TF, preds_TF = train_mbti_dimension(X, df['target_TF'], "T/F", "T (1)", "F (0)")
model_JP, scaler_JP, y_test_JP, preds_JP = train_mbti_dimension(X, df['target_JP'], "J/P", "J (1)", "P (0)")

total_elapsed = time.time() - total_start
print(f"\n{'='*55}")
print(f"✅ HOÀN TẤT! Tổng thời gian: {total_elapsed:.1f}s ({total_elapsed/60:.1f} phút)")
print(f"{'='*55}")


🔥 TRAINING VỚI TUNING NẶNG (200 iter, 10-fold, 3-model ensemble)
⏳ Mỗi trục mất ~2-5 phút. Tổng ~10-20 phút.



## 2.5. Model Evaluation (Báo cáo Phân loại & Confusion Matrix)
Để làm báo cáo, chúng ta cần xem chi tiết **F1-Score** (Độ chính xác chi tiết) và **Confusion Matrix** (Ma trận nhầm lẫn) của cả 4 mô hình.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

results = [
    ("E/I", y_test_EI, preds_EI, ["I", "E"]),
    ("S/N", y_test_SN, preds_SN, ["N", "S"]),
    ("T/F", y_test_TF, preds_TF, ["F", "T"]),
    ("J/P", y_test_JP, preds_JP, ["P", "J"])
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (name, y_true, y_pred, labels) in zip(axes.flatten(), results):
    print(f"\n{'='*40}\nBÁO CÁO TRỤC {name}")
    print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))
    
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=labels, yticklabels=labels)
    ax.set_title(f"Confusion Matrix: {name}", fontweight='bold')
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.tight_layout()
plt.show()


## 3. Phân tích Tầm quan trọng (Feature Importance)
Cùng xem chỉ số âm nhạc nào ảnh hưởng lớn nhất đến từng cục diện tính cách MBTI!


In [ ]:
# Lấy XGBoost từ Ensemble để xem Feature Importance
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
models_list = [
    (model_EI.named_estimators_['xgb'], 'E vs I'),
    (model_SN.named_estimators_['xgb'], 'S vs N'),
    (model_TF.named_estimators_['xgb'], 'T vs F'),
    (model_JP.named_estimators_['xgb'], 'J vs P')
]

for ax, (m, title) in zip(axes.flatten(), models_list):
    importance = m.feature_importances_
    sorted_idx = np.argsort(importance)[::-1][:8]
    
    feat_names = np.array(features)[sorted_idx]
    feat_vals = importance[sorted_idx]
    
    bars = ax.barh(range(len(feat_names)), feat_vals, color=plt.cm.viridis(np.linspace(0.3, 0.9, len(feat_names))))
    ax.set_yticks(range(len(feat_names)))
    ax.set_yticklabels(feat_names)
    ax.invert_yaxis()
    ax.set_title(f"Top 8 Features: {title}", fontweight='bold')
    
plt.tight_layout()
plt.show()


## 4. Kết quả cuối
Đưa 1 bài hát vào, cả 4 Models sẽ xúm lại bỏ phiếu, tạo ra 1 loại tính cách cuối cùng.


In [ ]:
def predict_full_mbti(test_features_1d):
    # Giả lập quy trình App Python: Truyền 1 mảng numpy chứa 10 features
    # Reshape về dạng 2D
    x_input = test_features_1d.values.reshape(1, -1)
    
    # Mô hình E/I
    pred_EI = model_EI.predict(scaler_EI.transform(x_input))[0]
    letter_1 = 'E' if pred_EI == 1 else 'I'
    
    # Mô hình S/N
    pred_SN = model_SN.predict(scaler_SN.transform(x_input))[0]
    letter_2 = 'S' if pred_SN == 1 else 'N'
    
    # Mô hình T/F
    pred_TF = model_TF.predict(scaler_TF.transform(x_input))[0]
    letter_3 = 'T' if pred_TF == 1 else 'F'
    
    # Mô hình J/P
    pred_JP = model_JP.predict(scaler_JP.transform(x_input))[0]
    letter_4 = 'J' if pred_JP == 1 else 'P'
    
    return f"{letter_1}{letter_2}{letter_3}{letter_4}"

# Test ngẫu nhiên 5 bài hát trong Data
sample_df = df.sample(5)
for idx, row in sample_df.iterrows():
    true_mbti = row['mbti_label']
    raw_features = row[features]
    predicted = predict_full_mbti(raw_features)
    
    icon = "✅" if predicted == true_mbti else "❌"
    print(f"{icon} Bài hát: {row['title']} | Thực tế: {true_mbti} | Model Đoán: {predicted}")
